In [ ]:
# Run this cell first in Colab
from google.colab import drive
drive.mount('/content/drive')

# Clone your repo
import os
repo_path = '/content/Brain_Tumor_Classification'

if not os.path.exists(repo_path):
    !git clone https://github.com/DevashreeNaidu/Brain_Tumor_Classification.git
    
os.chdir(repo_path)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
!pip install -q tensorflow numpy matplotlib scikit-learn pillow seaborn

# Add src to path so we can import our modules
import sys
sys.path.append('/content/Brain_Tumor_Classification')

# Set data paths for Colab
import os
TRAIN_DIR = '/content/drive/MyDrive/MS/Project Tumor/data/raw/Training'
TEST_DIR = '/content/drive/MyDrive/MS/Project Tumor/data/raw/Testing'
MODELS_DIR = '/content/drive/MyDrive/MS/Project Tumor/models'
FIGURES_DIR = '/content/drive/MyDrive/MS/Project Tumor/results/figures'

# Create directories if they don't exist
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Paths configured")
print(f"Training data: {TRAIN_DIR}")
print(f"Models will save to: {MODELS_DIR}")

In [ ]:
# Load and split dataset
from src.preprocessing import load_dataset, CLASSES
from sklearn.model_selection import train_test_split
import numpy as np

print("Loading training data... (this takes a few minutes)")
X_train_full, y_train_full = load_dataset(TRAIN_DIR)

print("Loading test data...")
X_test, y_test = load_dataset(TEST_DIR)

# Split training into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

print(f"\nTrain: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")
print("\nData ready.")

In [ ]:
# Run all experiments
from src.train import train_baseline, train_mobilenetv2, train_resnet50
from src.evaluate import evaluate_model, plot_training_history, summarize_all_experiments

results = []

# E1 — Baseline CNN, no augmentation
model_e1, history_e1 = train_baseline(X_train, y_train, X_val, y_val, augment=False)
plot_training_history(history_e1, 'E1 Baseline No Augmentation')
results.append(evaluate_model(model_e1, X_test, y_test, 'E1 Baseline No Augmentation'))

# E2 — Baseline CNN + augmentation
model_e2, history_e2 = train_baseline(X_train, y_train, X_val, y_val, augment=True)
plot_training_history(history_e2, 'E2 Baseline Augmentation')
results.append(evaluate_model(model_e2, X_test, y_test, 'E2 Baseline Augmentation'))

# E3 — MobileNetV2
model_e3, history_e3_head, history_e3_ft = train_mobilenetv2(X_train, y_train, X_val, y_val)
plot_training_history(history_e3_head, 'E3 MobileNetV2 Phase 1')
plot_training_history(history_e3_ft, 'E3 MobileNetV2 Phase 2')
results.append(evaluate_model(model_e3, X_test, y_test, 'E3 MobileNetV2'))

# E4 — ResNet50
model_e4, history_e4_head, history_e4_ft = train_resnet50(X_train, y_train, X_val, y_val)
plot_training_history(history_e4_head, 'E4 ResNet50 Phase 1')
plot_training_history(history_e4_ft, 'E4 ResNet50 Phase 2')
results.append(evaluate_model(model_e4, X_test, y_test, 'E4 ResNet50'))

# Summary
summarize_all_experiments(results)